In [1]:
# ensure classes imported from .py files are dynamically updated
%load_ext autoreload
%autoreload 2

# plot matplots nicely
%matplotlib inline  

In [2]:
from stericheight.data_handler import *
from stericheight.plotting_fns import PlottingFns
from stericheight.utils import Utils
from load_meop import MEOP
from region import Region
from composite import Composite
pfns = PlottingFns()

In [3]:
import gsw
import cmocean
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.optimize import root_scalar


folder = '../data/ARGO/'

In [5]:
ds1 = xr.open_dataset(folder + 'GL_PR_PF_5904096.nc')

In [6]:
import glob
glob.glob(folder + '*nc')

['../data/ARGO/GL_PR_PF_2902228.nc',
 '../data/ARGO/GL_PR_PF_2903867.nc',
 '../data/ARGO/GL_PR_PF_3902129.nc',
 '../data/ARGO/GL_PR_PF_5904096.nc',
 '../data/ARGO/GL_PR_PF_5904105.nc',
 '../data/ARGO/GL_PR_PF_5905065.nc',
 '../data/ARGO/GL_PR_PF_5905097.nc',
 '../data/ARGO/GL_PR_PF_5905233.nc',
 '../data/ARGO/GL_PR_PF_5905379.nc',
 '../data/ARGO/GL_PR_PF_5905638.nc',
 '../data/ARGO/GL_PR_PF_5905849.nc',
 '../data/ARGO/GL_PR_PF_5905881.nc',
 '../data/ARGO/GL_PR_PF_6903813.nc',
 '../data/ARGO/GL_PR_PF_6903814.nc',
 '../data/ARGO/GL_PR_PF_7900010.nc',
 '../data/ARGO/GL_PR_PF_7900011.nc',
 '../data/ARGO/GL_PR_PF_7900321.nc',
 '../data/ARGO/GL_PR_PF_7900322.nc',
 '../data/ARGO/GL_PR_PF_7900329.nc',
 '../data/ARGO/GL_PR_PF_7900330.nc',
 '../data/ARGO/GL_PR_PF_7900336.nc',
 '../data/ARGO/GL_PR_PF_7900604.nc',
 '../data/ARGO/GL_PR_PF_7900605.nc',
 '../data/ARGO/GL_PR_PF_7900606.nc',
 '../data/ARGO/GL_PR_PF_7900607.nc',
 '../data/ARGO/GL_PR_PF_7900608.nc',
 '../data/ARGO/GL_PR_PF_7900623.nc',
 

In [7]:
data = dict()
for file in glob.glob(folder + '*nc'):
    ds = xr.open_dataset(file)
    times=[]
    for t in ds.TIME:
        dst = ds.sel(TIME=t)
        if dst.TIME < pd.Timestamp(2022,1,1):
            if elevation.interp({'latitude':dst.LATITUDE,'longitude':dst.LONGITUDE}) > -1000:
                times.append(t)
    data[file] = times

In [8]:
profiles = []
pressure_axis = np.arange(0,1002,2)
for f in data:
    if data[f]:
        ds = xr.open_dataset(f)
        for t in data[f]:
            dst = ds.sel(TIME=t)
            dstn = dst.swap_dims({'DEPTH':'PRES'}).dropna(dim='PRES')
            if len(dstn.PRES)>0:
                profiles.append(dstn.interp({'PRES':pressure_axis}).expand_dims(['TIME','LATITUDE','LONGITUDE']))
        

In [9]:
xr.merge(profiles)

MemoryError: Unable to allocate 1.66 GiB for an array with shape (80, 74, 75, 501) and data type float64